# English–Vietnamese Speech API — Kaggle T4x2 Production Showcase

## ATTACH MODELS BEFORE RUN ALL

Configure the Kaggle notebook **before** running all cells:

- Accelerator: **GPU T4 x2**.
- Internet: enabled for repository/runtime bootstrap when needed.
- Attach **TranslateGemma 4B IT** so this path exists: `/kaggle/input/models/keras/translategemma/keras/translategemma_4b_it/<version>/`.
- Attach **Whisper Large V3 Turbo** mirror so this path exists: `/kaggle/input/models/dangkhoa2016/openai-whisper-large-v3-turbo/transformers/default/<version>/`.
- Attach **VoxCPM2** mirror so this path exists: `/kaggle/input/models/dangkhoa2016/openbmb-voxcpm2/pytorch/default/<version>/`.

The notebook intentionally keeps TranslateGemma on physical GPU 0 and Whisper + VoxCPM2 on physical GPU 1. Internal ports 8091/8092 stay localhost-only. The optional Cloudflare tunnel is disabled unless authentication is explicitly enabled.


In [ ]:
import json
import os
import subprocess
import time
import zipfile
from pathlib import Path

from IPython.display import Audio, display

# Evidence command: git rev-parse HEAD
REPO = Path('/kaggle/working/English-Vietnamese-Speech-API-On-Kaggle-T4x2-GPU')
REPO_URL = 'https://github.com/dangkhoa2016/English-Vietnamese-Speech-API-On-Kaggle-T4x2-GPU.git'
if not (REPO / '.git').is_dir():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO)], check=True)
os.chdir(REPO)
print('Repository:', REPO)
subprocess.run(['git', 'rev-parse', 'HEAD'], check=True)
subprocess.run(['git', 'status', '--short', '--branch'], check=True)


In [ ]:
required_model_roots = {
    'TranslateGemma 4B IT': Path('/kaggle/input/models/keras/translategemma/keras/translategemma_4b_it'),
    'Whisper Large V3 Turbo': Path('/kaggle/input/models/dangkhoa2016/openai-whisper-large-v3-turbo/transformers/default'),
    'VoxCPM2': Path('/kaggle/input/models/dangkhoa2016/openbmb-voxcpm2/pytorch/default'),
}
for name, path in required_model_roots.items():
    assert path.is_dir(), f'Missing attached model: {name}: {path}'
    print(f'ATTACHED: {name}: {path}')


In [ ]:
subprocess.run(['bash', 'scripts/bootstrap_kaggle.sh'], check=True)
print('Runtime versions:')
subprocess.run(['.venv/bin/python', '-c', "import jax,keras,keras_hub; print('jax',jax.__version__); print('keras',keras.__version__); print('keras_hub',keras_hub.__version__)"], check=True)
subprocess.run(['.venv-speech/bin/python', '-c', "import torch,transformers; import importlib.metadata as m; print('torch',torch.__version__); print('transformers',transformers.__version__); print('voxcpm',m.version('voxcpm'))"], check=True)


In [ ]:
env = os.environ.copy()
env['PYTHON_BIN'] = str(REPO / '.venv/bin/python')
subprocess.run(['bash', 'scripts/preflight.sh'], check=True, env=env)


In [ ]:
subprocess.run(['.venv/bin/ruff', 'check', 'src', 'tests'], check=True)
subprocess.run(['.venv/bin/ruff', 'format', '--check', 'src', 'tests'], check=True)
subprocess.run(['.venv/bin/python', '-m', 'pytest', '-q'], check=True)
print('CPU_TEST_GATE=PASS')


In [ ]:
subprocess.run(['bash', 'scripts/stop.sh'], check=True)
used = subprocess.check_output(['nvidia-smi','--query-gpu=memory.used','--format=csv,noheader,nounits'], text=True)
assert all(int(x.strip()) <= 64 for x in used.splitlines()), f'GPU baseline not clean: {used}'
service_env = os.environ.copy()
service_env.update({
    'SPEECH_BACKEND':'real',
    'TRANSLATION_BACKEND':'real',
    'API_AUTH_REQUIRED': os.getenv('NOTEBOOK_API_AUTH_REQUIRED', 'false'),
    'STARTUP_TIMEOUT_S':'360',
})
subprocess.run(['bash', 'scripts/start.sh'], check=True, env=service_env)
subprocess.run(['bash', 'scripts/status.sh'], check=True)
subprocess.run(['nvidia-smi'], check=True)


## Public transcription demo
The next cell calls only the public coordinator on port 8090.


In [ ]:
demo_dir = REPO / 'runtime/notebook-showcase'
demo_dir.mkdir(parents=True, exist_ok=True)
fixture = REPO / 'data/fixtures/english-short.wav'
transcription_json = demo_dir / 'transcription.json'
with transcription_json.open('wb') as out:
    subprocess.run(['curl','-fsS','-F',f'file=@{fixture};type=audio/wav','http://127.0.0.1:8090/v1/audio/transcriptions'], check=True, stdout=out)
transcription = json.loads(transcription_json.read_text())
print(json.dumps(transcription, ensure_ascii=False, indent=2))


## Public English and Vietnamese TTS demos
Both requests use the public `/v1/audio/speech` endpoint; the model worker remains isolated on physical GPU 1.


In [ ]:
tts_cases = {
    'en': 'Hello from the Kaggle T4x2 production showcase.',
    'vi': 'Xin chào từ bản trình diễn production trên Kaggle T4x2.',
}
for language, text in tts_cases.items():
    text_path = demo_dir / f'tts-{language}.txt'
    wav_path = demo_dir / f'tts-{language}.wav'
    text_path.write_text(text, encoding='utf-8')
    subprocess.run(['curl','-fsS','-F',f'file=@{text_path};type=text/plain','-F',f'language={language}','http://127.0.0.1:8090/v1/audio/speech','-o',str(wav_path)], check=True)
    print(language, wav_path, wav_path.stat().st_size, 'bytes')
    display(Audio(filename=str(wav_path)))


## Flagship English audio → Vietnamese speech
Exact stage order: normalize → Whisper → TranslateGemma → VoxCPM2 → ZIP.


In [ ]:
flagship_zip = demo_dir / 'flagship.zip'
subprocess.run(['curl','-fsS','-F',f'file=@{fixture};type=audio/wav','http://127.0.0.1:8090/v1/audio/translate-and-speak','-o',str(flagship_zip)], check=True)
extract_dir = demo_dir / 'flagship'
extract_dir.mkdir(exist_ok=True)
with zipfile.ZipFile(flagship_zip) as archive:
    names = set(archive.namelist())
    expected = {'transcript.en.txt','translation.vi.txt','speech.vi.wav','metadata.json'}
    assert names == expected, names
    archive.extractall(extract_dir)
transcript = (extract_dir/'transcript.en.txt').read_text(encoding='utf-8').strip()
translation = (extract_dir/'translation.vi.txt').read_text(encoding='utf-8').strip()
metadata = json.loads((extract_dir/'metadata.json').read_text(encoding='utf-8'))
print('EN:', transcript)
print('VI:', translation)
print(json.dumps(metadata, ensure_ascii=False, indent=2))
display(Audio(filename=str(extract_dir/'speech.vi.wav')))


## Bounded warm benchmark
The full retained qualification is documented in `docs/BENCHMARKS.md`. This notebook runs one bounded sequential request and intentionally skips the two-request stress test to keep the public showcase concise.


In [ ]:
bench_env = os.environ.copy()
bench_env.update({'SEQUENTIAL_RUNS':'1','RUN_CONCURRENT_QUALIFICATION':'false'})
subprocess.run(['bash', 'scripts/benchmark_pipeline.sh'], check=True, env=bench_env)
bench = json.loads((REPO/'runtime/benchmark-pipeline/results.json').read_text())
print(json.dumps({'sequential_summary':bench['sequential_summary'],'standalone':bench['standalone']}, ensure_ascii=False, indent=2))


## Optional authenticated Cloudflare Quick Tunnel
Disabled by default. To opt in, start the notebook with `NOTEBOOK_API_AUTH_REQUIRED=true` and set `ENABLE_CLOUDFLARE_TUNNEL=true`. Only coordinator port 8090 may be exposed.


In [ ]:
tunnel_proc = None
if os.getenv('ENABLE_CLOUDFLARE_TUNNEL', 'false').lower() == 'true':
    assert os.getenv('NOTEBOOK_API_AUTH_REQUIRED', 'false').lower() == 'true', 'Refusing tunnel without API auth'
    tunnel_env = os.environ.copy()
    tunnel_env.update({'ENABLE_CLOUDFLARE_TUNNEL':'true','API_AUTH_REQUIRED':'true'})
    tunnel_proc = subprocess.Popen(['bash','scripts/run_tunnel.sh'], env=tunnel_env)
    time.sleep(3)
    print('Tunnel process started. API key is stored in runtime/api_key.txt and is not printed by this notebook.')
else:
    print('Cloudflare tunnel disabled (default).')


## Clean stop and final evidence


In [ ]:
if tunnel_proc is not None and tunnel_proc.poll() is None:
    tunnel_proc.terminate()
    tunnel_proc.wait(timeout=10)
subprocess.run(['bash','scripts/stop.sh'], check=True)
time.sleep(2)
subprocess.run(['nvidia-smi','--query-gpu=index,name,memory.used','--format=csv,noheader,nounits'], check=True)
subprocess.run(['.venv/bin/python','-c',"from en_vi_speech_api.coordinator.lifecycle import ports_are_free; assert ports_are_free((8090,8091,8092)); print('PORT_CLEANUP=PASS')"], check=True)
print('KAGGLE_PRODUCTION_SHOWCASE=PASS')
